# Ders 18: Kriptografik Makbuzlarla AI Ajanlarını Güvence Altına Alma

## Uygulamalı Defter

Bu defter dört egzersiz üzerinden ilerler:

1. Bir ajan araç çağrısı için **ilk makbuzunuzu imzalayın** ve doğrulayın.
2. Makbuzda **hile yapın** ve doğrulamanın başarısız olduğunu izleyin.
3. **Üç makbuzluk bir zincir oluşturun** ve zincir bütünlüğünü onaylayın.
4. Bir Microsoft Agent Framework araç çağrısını **sararak her hareketin bir makbuz oluşturmasını sağlayın**.

Tüm kriptografik ilkelere iyi bakımı yapılan kütüphanelerden alınır (`pynacl` Ed25519 için, `jcs` RFC 8785 kipli JSON için, Python standart kütüphanesinden `hashlib` SHA-256 için). Makbuz mantığı kendisi basit Python’dur ve okuyup değiştirebilirsiniz.

Hücreleri sırayla çalıştırın. Her bölüm kısa ve kendi içinde tamamlanmıştır.


## Kurulum

İki bağımlılığı yükleyin. İkisi de izin verici lisanslara sahiptir (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Yardımcı Araçlar

Bu iki yardımcı, base64url kodlamasını (dolgu olmadan) ve rasgele nesnelerin SHA-256 karmasını işler. Geri kalan not defterinin yalnızca makbuz mantığına odaklanmasını sağlarlar.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Bölüm 1: İlk makbuzunuzu imzalayın

Diyelim ki **Contoso Travel** için görevli, bir müşteri için Sydney'den Los Angeles'a uçuşlara baktı. Bu araç çağrısını, gelecekteki bir denetçinin bize güvenmeden doğrulayabilmesi için imzalanmış bir makbuz olarak kaydetmek istiyoruz.

### Adım 1.1: Bir imzalama anahtarı oluşturun

Üretimde, görevlisinin imzalama anahtarı bir donanım güvenlik modülünde (HSM), Azure Key Vault'ta veya benzer korumalı bir depoda bulunur. Bu ders için bellekte taze bir anahtar oluşturuyoruz.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Adım 1.2: Makbuz içeriğini oluşturma

İçerik, makbuzun doğrulamasını istediğimiz her şeyi içerir: kim hareket etti, hangi araç kullanıldı, hangi argümanlarla, ne döndü, hangi politika kapsamında ve ne zaman. Argümanları ve sonucu doğrudan dahil etmek yerine, makbuzun hassas içeriği sızdırmaması için bunları karmalarını alıyoruz.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### 1.3. Adım: Makbuzu imzala ve birleştir

Üç adım:

1. İki uygulamanın da aynı mantıksal makbuzu üretmesi için JCS kullanarak yükü kanonik hale getirin, böylece baytlar tamamen aynı olur.
2. Kaniyonik baytların SHA-256 ile özetini alın.
3. Özeti Ed25519 özel anahtarıyla imzalayın.

İmza daha sonra orijinal yükle iliştirilir ve böylece nihai makbuz oluşturulur.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Adım 1.4: Makbuzu doğrulayın

Doğrulama işlemi süreci tersine çevirir. İmza çıkarılır, kanonik karmayı yeniden hesaplarız ve imzayı makbuzdaki açık anahtara karşı kontrol ederiz.

Bu doğrulamayı yapan bir denetçi bizden sadece makbuza ihtiyaç duyar. Çağrılacak bir hizmet yok, sorgulanacak bir anahtar dizini yok, güven gerekmiyor.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

`Makbuz geçerlidir: True` görmelisiniz. Ajan, ilk kriptografik olarak imzalanmış denetim kaydını oluşturdu.


## Bölüm 2: Makbuzu Değiştirme

Makbuzların temel amacı, müdahaleye karşı korumalı olmalarıdır. Bunu kanıtlayalım.

Makbuzun tam olarak bir karakterini değiştireceğiz ve doğrulamanın başarısız olduğunu gözlemleyeceğiz.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Ne oldu az önce?

`policy_id` değerini değiştirdiğimizde, kanonik baytlar değişti. Bu baytların SHA-256 karması değişti. İmza (orijinal karmaya yönelik olan) artık yeni karma ile eşleşmiyor. Doğrulama doğru şekilde `False` döndürüyor.

Makbuzdaki herhangi bir alanı değiştirmek ve yine de doğrulanmasını sağlamak mümkün değil, saldırganın özel anahtara sahip olmadığı sürece. Özel anahtar bir anahtar kasasında olduğu ve genel anahtar yayımlandığı sürece, tahrifat gizlenemez.

Kendiniz deneyin: Yukarıdaki hücrede `tool_name` veya `agent_id` ya da `timestamp` öğesini değiştirip yeniden çalıştırın. Her değişiklik geçersiz bir makbuz üretir.


## Bölüm 3: Makbuzları zincirleyin

Tek bir makbuz bir işlemi korur. Çoğu ajan birçok işlem yapar. Tüm dizinin müdahaleye karşı korumalı olmasını sağlamak için, her makbuzu önceki makbuza bağlarız; önceki makbuzun karmasını yeni makbuzun içeriğine dahil ederiz.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Eğer biri bir makbuzu çıkarır veya sırasını değiştirirse, zincir tam da o noktada kopar. Sonraki herhangi bir makbuzun doğrulanması başarısız olur çünkü `previous_receipt_hash` artık öncekinin gerçek karmasıyla eşleşmez.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Şimdi zinciri, ortadaki makbuzu değiştirmek suretiyle kırın ve yeniden doğrulayın. Değiştirilmiş makbuz imza kontrolünden geçemez ve sonraki makbuz da zincir bağlantısı kontrolünden geçemez (çünkü `previous_receipt_hash` artık değiştirilmiş ortadaki makbuzun hash'i ile eşleşmemektedir).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Makbuz 0 hala doğrulanıyor (değiştirilmedi ve bağlı olduğu önceki bir makbuz yok). Makbuz 1, `tool_args_hash` değiştirdiğimiz için imza kontrolünden geçemiyor. Makbuz 2 ise zincir-bağlantı kontrolünden geçemiyor çünkü `previous_receipt_hash` orijinal (şimdi değiştirilmiş) makbuz 1'e karşı hesaplandı.

Bir saldırgan değiştirilmiş makbuz 1'i yeniden imzalamaya kalksa bile (bunu özel anahtar olmadan yapamaz), makbuz 2’deki zincir-bağlantı uyumsuzluğu müdahalenin ortaya çıkmasını engelleyemez. Değişikliği gizlemek için saldırganın, modifikasyon noktasından itibaren her makbuzu yeniden imzalaması gerekir; bu da özel anahtarın kendisinde olması gerektiği anlamına gelir.


## Bölüm 4: Bir ajan araç çağrısını makbuz imzalama ile sarma

Gerçek bir dağıtımda, her ajan yazarının `make_receipt` çağrısını hatırlamasını istemezsiniz. Makbuz imzalama her araç çağrısı için otomatik olmalıdır.

İşte en basit kalıp: herhangi bir çağrılabilir araç fonksiyonunu alan ve onu makbuz yayımlayan bir sürüme dönüştüren bir sarıcı sınıfı. Bu, Microsoft Agent Framework (`agent_framework.azure`) dahil herhangi bir ajan çerçevesine uyarlanır.

Bir Azure AI Foundry projeniz yoksa, aşağıdaki yerel taklit yine de kalıbı gösterir.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Microsoft Agent Framework ile Entegrasyon

Yukarıdaki `ReceiptedTool` sarmalayıcı framework bağımsızdır. Microsoft Agent Framework ile oluşturulmuş bir ajan içinde kullanmak için, sarmalanmış işlevi bir araç olarak kaydedin. Bir taslak (sahteyi gerçek bir Azure AI Foundry araç kaydıyla değiştireceksiniz):

```python
# Entegrasyon şeklini gösteren sözde kod.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     talimatlar="Siz bir Contoso Seyahat acentesisiniz ...",
#     araçlar=[receipted_lookup],   # sarılmış araç, ham fonksiyon değil
# )
# yanıt = agent.run("Haziran ayında Sydney'den Los Angeles'a uçuşları bulun.")
#
# # Çalıştırma sonrası, ajan tarafından yapılan her araç çağrısının imzalı bir makbuzu vardır:
# audit_chain = receipted_lookup.receipts
```

Ajan framework'ün makbuzlar hakkında herhangi bir şey bilmesine gerek yoktur. Makbuz imzalama, framework'e bağlanmak yerine aracın etrafına sarılmıştır. Bu, ajanı yeniden yazmadan mevcut ajan koduna kaynak bilgisi eklemenin yoludur.


## Özet ve ileri seviye zorluk

Siz:

- Bir Ed25519 anahtar çifti oluşturdunuz.
- Bir aracın aracı çağrısı için makbuz oluşturup imzaladınız.
- Makbuzu yalnızca genel anahtarı kullanarak çevrimdışı doğruladınız.
- Bir makbuzu bozup doğrulama hatası gözlemlediniz.
- Üç makbuzdan oluşan bir karma zinciri oluşturdu.
- Zincirin ortasını bozdunuz ve hem imza hatası hem de zincir bağlantısı hatası gözlemlediniz.
- Bir araç fonksiyonunu otomatik makbuz imzalama ile sardınız.

**İleri seviye zorluk.** Makbuz şemasına bir `request_id` alanı (dağıtık izleme için UUID) ekleyin. `make_receipt` fonksiyonunu güncelleyip bunu dahil edin ve makbuzların uçtan uca hala doğrulandığını teyit edin. Sonra imzalama işleminden sonra alanı değiştirip doğrulamanın başarısız olduğunu teyit edin. Bu, her baytın kanonik kodlamada imzaya nasıl katkıda bulunduğunu içselleştirmenizi sağlar.

**Önemli sınır.** Makbuzlar üç şeyi kanıtlar ve sadece üç şeyi kanıtlar: atıf (bu anahtar bu içeriği imzaladı), bütünlük (içerik imzalandığından beri değişmedi) ve sıralama (bu makbuz o makbuzdan sonra geldi). Makbuzlar aracın eyleminin doğru olduğunu, `policy_id` ile adlandırılan politikanın gerçekten değerlendirildiğini veya aracın her kurala uyduğunu kanıtlamaz. Makbuzlar temeldir. Yönetim ise üzerine kurduğunuz sistemdir.

Bu sınırı aklınızda tutarak dersin README’sini tekrar okuyun. Takımların makbuzlarla ilgili en yaygın hatası, "makbuzlarımız var" demenin "yönetiliyoruz" anlamına geldiğini varsaymalarıdır. Böyle değildir. Makbuzlar aracın davranışını denetlenebilir kılar. Doğru olduğunu garanti etmez.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba sarf etsek de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal belge, kendi dilinde yetkili kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi önerilir. Bu çevirinin kullanımı sonucu ortaya çıkabilecek yanlış anlamalardan veya yanlış yorumlamalardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
